# Gridded CF NetCDF files

CF-convention rectilinear grids. For each file we open it, inspect dimensions/variables/attributes, plot a variable, retrieve the array, reduce a dimension, add and remove a variable, crop every variable with a single polygon, and save the result.

In [ ]:
%matplotlib inline
import tempfile
from pathlib import Path

import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon

from pyramids.feature import FeatureCollection
from pyramids.netcdf import NetCDF

DATA = Path('../../../../examples/data/netcdf/samples')

## `cf__7v__1d3-2d3-3d1__y-asc.nc`

Sea-surface temperature on a global CF grid with **non-square cells** (2° longitude, 1° latitude). Exercises the full workflow — inspect, plot, retrieve, reduce, mutate, wrap longitude, and crop with a polygon — on a grid whose lon and lat spacings differ.

**Open the file and inspect the container**

In [ ]:
nc = NetCDF.read_file(DATA / 'cf__7v__1d3-2d3-3d1__y-asc.nc')
nc

**Dimensions and variables**

In [ ]:
# get_all_metadata() returns a NetCDFMetadata whose summary lists dimensions and, for each
# variable, its dims / shape / dtype / unit / scale-offset.
meta = nc.get_all_metadata()
print(meta)

**Global attributes**

In [ ]:
nc.global_attributes

**Plot the variable** (a 2-D slice is auto-selected for >2-D variables)

In [ ]:
# select the variable to plot
field = nc.get_variable('tos')
# this grid's longitudes run 0..360; wrap them to -180..180 so it lines up with the coastline
field = field.wrap_longitude()
# plot in the data's own CRS (lon/lat degrees) — no reprojection
glyph = field.plot()
# draw the Natural Earth coastline on top so the geography stays visible over the data
glyph.add_features("coastline", "50m", zorder=5)

**Wrap longitude to −180–180** — this grid spans 0–360 (Pacific-centred). `wrap_longitude` re-frames it to the −180–180 (Greenwich-centred) convention; we wrap the selected variable, then plot the re-centred map.

In [ ]:
# select the variable to plot
field = nc.get_variable('tos')
# this grid's longitudes run 0..360; wrap them to -180..180 so it lines up with the coastline
field = field.wrap_longitude()
# plot in the data's own CRS (lon/lat degrees) — no reprojection
glyph = field.plot()
# draw the Natural Earth coastline on top so the geography stays visible over the data
glyph.add_features("coastline", "50m", zorder=5)

**Retrieve the underlying data**

In [ ]:
var = nc.get_variable('tos')
data = var.read_array()
print('shape:', data.shape)
print(
    'min / mean / max:',
    float(np.nanmin(data)),
    float(np.nanmean(data)),
    float(np.nanmax(data)),
)

**Reduce a dimension** — collapse the time axis to its mean

In [ ]:
time_mean = nc.reduce('time', how='mean')
print('dimensions after reducing time:', dict(time_mean.dimension_sizes))

**Add a variable** — derive a 2-D field and append it as a new variable

In [ ]:
work = Path(tempfile.mkdtemp())
slice2d = data[tuple(0 for _ in range(data.ndim - 2))]
NetCDF.create_from_array(
    arr=slice2d,
    geo=var.geotransform,
    epsg=var.epsg or 4326,
    variable_name='tos_slice0',
    path=str(work / 'derived.nc'),
)
nc.add_variable(NetCDF.read_file(str(work / 'derived.nc')), 'tos_slice0')
print('variables after add:', nc.variable_names)

**Remove a variable**

In [ ]:
nc.remove_variable('tos_slice0')
print('variables after remove:', nc.variable_names)

**Crop with a real-world polygon** — because `tos` is *sea-surface* temperature, we crop to a recognisable body of water: the **Gulf of Mexico and Caribbean Sea**. This grid uses 0-360 longitudes, so we `wrap_longitude()` the variable to -180..180 to line it up with the polygon, then crop and plot it in its own lon/lat CRS with the Natural Earth coastline drawn on top. The crop keeps this grid's non-square **2°×1°** cells.

In [ ]:
# Gulf of Mexico + Caribbean Sea, in -180..180 longitudes
region = [
    (-98, 18),
    (-93, 30),
    (-80, 31),
    (-75, 27),
    (-63, 22),
    (-60, 16),
    (-65, 10),
    (-75, 9),
    (-84, 8),
    (-92, 11),
    (-98, 18),
]
aoi = FeatureCollection(gpd.GeoDataFrame(geometry=[Polygon(region)], crs=4326))

# select the sea-surface-temperature variable
tos = nc.get_variable('tos')
# this grid is 0..360; wrap to -180..180 so it matches the polygon (defined in -180..180)
tos = tos.wrap_longitude()
# crop to the polygon (cells outside it become no-data)
tos_gulf = tos.crop(aoi)
print('cropped tos bounds:', [round(b, 1) for b in tos_gulf.total_bounds])
print('cell sizes (x, y):', tos_gulf.geotransform[1], tos_gulf.geotransform[5])
# plot the cropped sea-surface temperature in its own lon/lat CRS
glyph = tos_gulf.plot()
# draw the Natural Earth coastline on top so the geography stays visible over the data
glyph.add_features("coastline", "50m", zorder=5)

**Crop the whole container** — `nc.crop` clips **every** variable in one call and returns a new `NetCDF`. This runs in the file's native 0–360 longitudes (the polygon is shifted back by +360); we then re-display the Gulf-cropped `tos` variable in its own lon/lat CRS with a coastline overlay.

In [ ]:
# the whole-container crop runs in the file's native 0-360 longitudes, so shift the polygon by +360
native_region = [(x + 360 if x < 0 else x, y) for x, y in region]
aoi_native = FeatureCollection(
    gpd.GeoDataFrame(geometry=[Polygon(native_region)], crs=4326)
)
# crop every variable in the container in one call
cropped = nc.crop(aoi_native)
print('variables in cropped container:', cropped.variable_names)
glyph = tos_gulf.plot()
# draw the Natural Earth coastline on top so the geography stays visible over the data
glyph.add_features("coastline", "50m", zorder=5)

**Save the result to a new NetCDF file**

In [ ]:
out = work / 'cropped.nc'
cropped.to_file(out)
print('saved cropped container to', out.name, '->', out.exists())

## `cf__12v__1d4-2d5-3d2-4d1__y-asc.nc`

Multi-variable CF file with a 4-D variable (time, level, lat, lon) plus bounds. Its latitudes are Gaussian, so it is not affine-croppable.

**Open the file and inspect the container**

In [ ]:
nc = NetCDF.read_file(DATA / 'cf__12v__1d4-2d5-3d2-4d1__y-asc.nc')
nc

**Dimensions and variables**

In [ ]:
# get_all_metadata() returns a NetCDFMetadata whose summary lists dimensions and, for each
# variable, its dims / shape / dtype / unit / scale-offset.
meta = nc.get_all_metadata()
print(meta)

**Global attributes**

In [ ]:
nc.global_attributes

**Plot the variable** (a 2-D slice is auto-selected for >2-D variables)

In [ ]:
# select the variable to plot
field = nc.get_variable('ua')
# this grid's longitudes run 0..360; wrap them to -180..180 so it lines up with the coastline
field = field.wrap_longitude()
# plot in the data's own CRS (lon/lat degrees) — no reprojection
glyph = field.plot()
# draw the Natural Earth coastline on top so the geography stays visible over the data
glyph.add_features("coastline", "50m", zorder=5)

**Retrieve the underlying data**

In [ ]:
var = nc.get_variable('ua')
data = var.read_array()
print('shape:', data.shape)
print(
    'min / mean / max:',
    float(np.nanmin(data)),
    float(np.nanmean(data)),
    float(np.nanmax(data)),
)

**Add a variable** — derive a 2-D field and append it as a new variable

In [ ]:
work = Path(tempfile.mkdtemp())
slice2d = data[tuple(0 for _ in range(data.ndim - 2))]
NetCDF.create_from_array(
    arr=slice2d,
    geo=var.geotransform,
    epsg=var.epsg or 4326,
    variable_name='ua_slice0',
    path=str(work / 'derived.nc'),
)
nc.add_variable(NetCDF.read_file(str(work / 'derived.nc')), 'ua_slice0')
print('variables after add:', nc.variable_names)

**Remove a variable**

In [ ]:
nc.remove_variable('ua_slice0')
print('variables after remove:', nc.variable_names)

> **Note** — this file has heterogeneous variable dimensions and a non-affine grid (Gaussian/spectral latitudes, so no single affine geotransform). Container-wide operations — `reduce` across one shared dimension and polygon `crop` (an affine warp) — do not apply uniformly here, so we focus on per-variable inspection, plotting, retrieval, and mutation, then save the container.

**Save the container to a new NetCDF file**

In [ ]:
out = work / 'saved.nc'
nc.to_file(out)
print('saved container to', out.name, '->', out.exists())

## `cf__48v__1d17-3d21-4d10__y-asc.nc`

Large multi-variable CF file mixing 3-D and 4-D fields and string variables on a spectral (CAM) grid. `T` is stored as `(time, lat, lev, lon)` — lat/lon are not the trailing dims — so it needs explicit `x_dim`/`y_dim`; it is also not affine-croppable.

**Open the file and inspect the container**

In [ ]:
nc = NetCDF.read_file(DATA / 'cf__48v__1d17-3d21-4d10__y-asc.nc')
nc

**Dimensions and variables**

In [ ]:
# get_all_metadata() returns a NetCDFMetadata whose summary lists dimensions and, for each
# variable, its dims / shape / dtype / unit / scale-offset.
meta = nc.get_all_metadata()
print(meta)

**Global attributes**

In [ ]:
nc.global_attributes

**Plot the variable** — this file stores its dimensions in a non-standard order and its lat/lon coordinates carry no CF axis attributes, so CF auto-detection cannot find the geographic plane. Pass `x_dim`/`y_dim` to select it explicitly (without them, the last two dimensions would be used, giving a level–longitude cross-section).

In [ ]:
# select the variable to plot
field = nc.get_variable('T', x_dim='lon', y_dim='lat')
# this grid's longitudes run 0..360; wrap them to -180..180 so it lines up with the coastline
field = field.wrap_longitude()
# plot in the data's own CRS (lon/lat degrees) — no reprojection
glyph = field.plot()
# draw the Natural Earth coastline on top so the geography stays visible over the data
glyph.add_features("coastline", "50m", zorder=5)

**Retrieve the underlying data**

In [ ]:
var = nc.get_variable('T', x_dim='lon', y_dim='lat')
data = var.read_array()
print('shape:', data.shape)
print(
    'min / mean / max:',
    float(np.nanmin(data)),
    float(np.nanmean(data)),
    float(np.nanmax(data)),
)

**Add a variable** — derive a 2-D field and append it as a new variable

In [ ]:
work = Path(tempfile.mkdtemp())
slice2d = data[tuple(0 for _ in range(data.ndim - 2))]
NetCDF.create_from_array(
    arr=slice2d,
    geo=var.geotransform,
    epsg=var.epsg or 4326,
    variable_name='T_slice0',
    path=str(work / 'derived.nc'),
)
nc.add_variable(NetCDF.read_file(str(work / 'derived.nc')), 'T_slice0')
print('variables after add:', nc.variable_names)

**Remove a variable**

In [ ]:
nc.remove_variable('T_slice0')
print('variables after remove:', nc.variable_names)

> **Note** — this file has heterogeneous variable dimensions and a non-affine grid (Gaussian/spectral latitudes, so no single affine geotransform). Container-wide operations — `reduce` across one shared dimension and polygon `crop` (an affine warp) — do not apply uniformly here, so we focus on per-variable inspection, plotting, retrieval, and mutation, then save the container.

**Save the container to a new NetCDF file**

In [ ]:
out = work / 'saved.nc'
nc.to_file(out)
print('saved container to', out.name, '->', out.exists())